# Phase 1.10: Triage Classifier

In [1]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import json
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet('exports/processed_violations.parquet')

# Labeled data
labeled = df[df.validation_status.isin(['approved', 'rejected'])].copy()
labeled['is_approved'] = (labeled.validation_status == 'approved').astype(int)

print(f"Total labeled records: {len(labeled)}")
print(labeled['is_approved'].value_counts(normalize=True))


Total labeled records: 165154
is_approved
1    0.698742
0    0.301258
Name: proportion, dtype: float64


In [2]:
labeled['device_mode_enc'] = (labeled['device_mode'] == 'MOBILE').astype(int)

features = [
    'hour_ist', 'dow', 'is_weekend', 'vehicle_footprint',
    'severity_score', 'violation_count', 'has_junction',
    'device_mode_enc', 'repeat_plate'
]

X = labeled[features]
y = labeled['is_approved']

# Fast training for one fold to get metrics
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(skf.split(X, y))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model = XGBClassifier(eval_metric='logloss')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)
print(f"Precision/recall/AUC on held-out fold. AUC: {auc:.4f}")

# Export feature importance & metrics
fi = dict(zip(features, map(float, model.feature_importances_)))
metrics = {
    'auc': float(auc),
    'feature_importances': fi,
    'confusion_matrix': confusion_matrix(y_test, y_pred).tolist()
}
with open('exports/triage_metrics.json', 'w') as f:
    json.dump(metrics, f)
print("Saved triage_metrics.json")


Precision/recall/AUC on held-out fold. AUC: 0.6318
Saved triage_metrics.json
